In [3]:
from mylib.statistic_test import *
from mazepy.datastruc.neuact import SpikeTrain, TuningCurve
from mazepy.datastruc.variables import VariableBin

code_id = "0877 - Revisit Stability"
loc = join(figpath, "Dsp", code_id)
mkdir(loc)

pass

        D:\Data\FinalResults\Dsp\0877 - Revisit Stability is already existed!


In [ ]:
def get_prev_rts(rt: int) -> np.ndarray[np.int64]:
    rt_order = np.array([0,4,1,5,2,6,3])
    idx = np.where(rt_order == rt)[0][0]
    return rt_order[:idx]

def determine_preretrieval_range(trace, rt: int, thre = 0.2) -> np.ndarray[np.int64]:
    """Return bins that are in the preretrieval range"""
    assert rt != 0, "RT 0 does not have preretrieval range"
    rt_to_nodes = [None, 1, 2, 3, 6, 7, 8]
    targ_nodes = 0 if rt in [1, 2, 3] else 5
    rt_nodes = rt_to_nodes[rt]
    
    map_targ = trace[f'node {targ_nodes}']['old_map_clear']
    map_rt = trace[f'node {rt_nodes}']['old_map_clear']
    
    corrs = np.zeros(CP_DSPs[1][rt].shape[0], dtype=np.float64)
    for i, b in enumerate(CP_DSPs[1][rt]):
        corrs[i] = np.corrcoef(map_targ[:, b-1], map_rt[:, b-1])[0, 1]
    
    return np.where(corrs <= thre)[0]
        
def get_lapwise_ratemap(trace: dict, is_cutted: bool = False, cutted_for_rt: int = 0):    
    maze_type = trace['maze_type']
    beg_time, end_time = trace['lap beg time'], trace['lap end time']
    beg_idx = np.array([np.where(trace['correct_time'] >= beg_time[i])[0][0] for i in range(beg_time.shape[0])])
    routes = classify_lap(spike_nodes_transform(trace['correct_nodes'], 12), beg_idx, maze_type)
    smoothed_map = np.zeros((trace['n_neuron'], 144, beg_idx.shape[0]), dtype = np.float64)
    
    for i in range(beg_idx.shape[0]):
        
        spike_idx = np.where(
            (trace['ms_time'] >= beg_time[i]) & (trace['ms_time'] <= end_time[i]) &
            (np.isnan(trace['spike_nodes_original']) == False)
        )[0]
        
        spike_nodes = spike_nodes_transform(trace['spike_nodes_original'][spike_idx].astype(np.int64), 12)-1
        Spikes = trace['Spikes_original'][:, spike_idx]
        
        spike_train = SpikeTrain(
            activity=Spikes,
            time=trace['ms_time'][spike_idx],
            variable=VariableBin(spike_nodes)
        )
        
        rate_map = spike_train.calc_tuning_curve(144, t_interv_limits=100)
        smoothed_map[:, :, i] = rate_map.to_array() #@ trace['Ms'].T
    
    if is_cutted:
        bins = CP_DSPs[maze_type][cutted_for_rt]-1
        smoothed_map = smoothed_map[:, bins, :]

    X = np.transpose(smoothed_map, (2, 1, 0))
    return X

def get_index_map(mouse: int) -> np.ndarray:
    idx = np.where(f_CellReg_dsp['MiceID'] == mouse)[0][0]
    with open(f_CellReg_dsp['cellreg_folder'][idx], 'rb') as f:
        index_map = pickle.load(f).astype(np.int64)
        
    if mouse != 10132:
        index_map = index_map[1:, :]
    
    
    return index_map

